# BDLegalRAG — Corpus Layer (Stage 1 & 2)
### BLAD (Bangladesh Legal Acts Dataset) থেকে শুরু করে cleaned, structured corpus তৈরি

এই notebook-টা একদম শুরু থেকে চলবে — environment setup, file upload, extraction, quality check, domain filtering, section parsing, corpus statistics, এবং final cleaned corpus save করা পর্যন্ত।

**Colab এবং Kaggle দুটোতেই চলবে** — নিচের cell গুলোতে দুই environment-এর জন্য আলাদা instruction দেওয়া আছে, যেটা relevant সেটা uncomment/run করুন।


## ধাপ ০: Environment Setup

প্রথমে ঠিক করুন আপনি **Colab** নাকি **Kaggle**-এ আছেন, এবং সেই অনুযায়ী storage mount করুন।


In [ ]:
# Detect environment
import sys

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab" if IN_COLAB else "Running in Kaggle / local — adjust paths below if needed")


Running in Colab


In [ ]:
# --- COLAB ONLY: mount Google Drive ---
# যদি Kaggle-এ থাকেন, এই cell স্কিপ করুন (Kaggle-এ Drive লাগবে না, নিজস্ব /kaggle/working আছে)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Set up working directories
import os

if IN_COLAB:
    BASE_DIR = "/content/drive/MyDrive/BDLegalRAG"
else:
    # Kaggle notebooks: use /kaggle/working (persists as notebook output)
    BASE_DIR = "/kaggle/working/BDLegalRAG"

RAW_DIR = os.path.join(BASE_DIR, "raw_data")
PROCESSED_DIR = os.path.join(BASE_DIR, "processed_data")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)


BASE_DIR: /content/drive/MyDrive/BDLegalRAG
RAW_DIR: /content/drive/MyDrive/BDLegalRAG/raw_data
PROCESSED_DIR: /content/drive/MyDrive/BDLegalRAG/processed_data


In [ ]:
# Install/verify required libraries (all lightweight, CPU-only — no GPU needed for this stage)
!pip install pandas --quiet
import json, re, zipfile, io
import pandas as pd
from collections import Counter

print("Libraries ready.")


Libraries ready.


## ধাপ ১: BLAD Dataset Upload করুন

আপনার কাছে থাকা BLAD zip ফাইলটা (`archive.zip` বা যে নামেই থাকুক) আপলোড করুন।

- **Colab**: নিচের cell চালান, একটা upload button আসবে
- **Kaggle**: zip-টা আগে থেকেই "Add Data" দিয়ে input হিসেবে যোগ করে রাখুন, তারপর নিচের cell স্কিপ করে সরাসরি `ZIP_PATH` variable-টা `/kaggle/input/.../archive.zip`-এ set করুন


# --- COLAB: file upload widget ---
if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()   # এখানে আপনার BLAD zip ফাইলটা বেছে নিন
    ZIP_PATH = list(uploaded.keys())[0]
    print("Uploaded:", ZIP_PATH)
else:
    # --- KAGGLE: set this to your added dataset's path ---
    # উদাহরণ: ZIP_PATH = "/kaggle/input/bangladesh-legal-acts-dataset/archive.zip"
    ZIP_PATH = "/kaggle/input/YOUR-DATASET-FOLDER/archive.zip"  # <-- এটা বদলে দিন
    print("Set ZIP_PATH manually above to your Kaggle input path.")

In [ ]:
# --- COLAB: file upload widget ---
if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()   # এখানে আপনার BLAD zip ফাইলটা বেছে নিন
    ZIP_PATH = list(uploaded.keys())[0]
    print("Uploaded:", ZIP_PATH)
else:
    # --- KAGGLE: set this to your added dataset's path ---
    # উদাহরণ: ZIP_PATH = "/kaggle/input/bangladesh-legal-acts-dataset/archive.zip"
    ZIP_PATH = "/kaggle/input/YOUR-DATASET-FOLDER/archive.zip"  # <-- এটা বদলে দিন
    print("Set ZIP_PATH manually above to your Kaggle input path.")

Saving archive.zip to archive.zip
Uploaded: archive.zip


In [ ]:
# Extract the archive into RAW_DIR
EXTRACT_DIR = os.path.join(RAW_DIR, "blad_extracted")
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_DIR)

print("Extracted to:", EXTRACT_DIR)
print("Contents:", os.listdir(EXTRACT_DIR))


Extracted to: /content/drive/MyDrive/BDLegalRAG/raw_data/blad_extracted
Contents: ['Contextualized_Bangladesh_Legal_Acts.json', 'acts', 'balanced_1000.csv', 'bangladesh_legal_systems.json', 'filtered_act_list.csv', 'govt.json']


## ধাপ ২: Data Quality Check — Sanity Check on Structure

extract হওয়া ফাইলগুলো ঠিকমতো আছে কিনা, একটা sample Act দেখে চেক করুন।


In [ ]:
ACTS_DIR = os.path.join(EXTRACT_DIR, "acts")
act_files = sorted(os.listdir(ACTS_DIR))
print(f"Total act files found: {len(act_files)}")

# Load one sample act and inspect
with open(os.path.join(ACTS_DIR, act_files[0]), encoding="utf-8") as f:
    sample = json.load(f)

print("\nKeys in each act record:")
print(list(sample.keys()))

print("\nSample act title:", sample.get("act_title"))
print("Sample act year:", sample.get("act_year"))
print("Number of sections:", len(sample.get("sections", [])))
print("\nFirst section text (raw):")
print(sample["sections"][0]["section_content"][:300] if sample.get("sections") else "NO SECTIONS FOUND")


Total act files found: 1484

Keys in each act record:
['act_title', 'act_no', 'act_year', 'publication_date', 'sections', 'footnotes', 'copyright_info', 'source_url', 'fetch_timestamp', 'csv_metadata', 'token_count', 'language', 'government_context', 'processing_info', 'legal_system_context']

Sample act title: 1The Districts Act, 1836
Sample act year: 1836
Number of sections: 1

First section text (raw):
It shall be lawful for the2[***] Government, by notification in the official Gazette, to create new districts in any part of Bangladesh.3[Government may, from time to time, alter the limits of the existing districts in any part of Bangladesh].


In [ ]:
# Check for any empty/malformed act files before proceeding — catch problems early
empty_sections = []
load_errors = []

for fname in act_files:
    path = os.path.join(ACTS_DIR, fname)
    try:
        with open(path, encoding="utf-8") as f:
            act = json.load(f)
        if not act.get("sections"):
            empty_sections.append(fname)
    except Exception as e:
        load_errors.append((fname, str(e)))

print(f"Acts with zero sections: {len(empty_sections)}")
print(f"Acts that failed to load: {len(load_errors)}")
if load_errors:
    print("First few errors:", load_errors[:5])


Acts with zero sections: 203
Acts that failed to load: 0


## ধাপ ৩: Act Index Load করুন এবং Repealed Act নিয়ে সিদ্ধান্ত নিন

`filtered_act_list.csv`-এ প্রতিটা Act-এর metadata আছে, যার মধ্যে `is_repealed` column আছে।
এখানে আমরা **repealed (বাতিল) Act বাদ দিচ্ছি** — কারণ বর্তমান আইনি প্রশ্নের উত্তরের জন্য কার্যকর আইনই relevant।
(চাইলে এই ফিল্টার বন্ধ করে সব রাখতে পারেন — নিচে `EXCLUDE_REPEALED` ভ্যারিয়েবল বদলান।)


In [ ]:
act_index_path = os.path.join(EXTRACT_DIR, "filtered_act_list.csv")
act_index = pd.read_csv(act_index_path)

print("Total acts in index:", len(act_index))
print(act_index.columns.tolist())
print(act_index.head(3))


Total acts in index: 1489
['Act Title', 'Act No', 'Act Year', 'Act Link', 'full_text', 'is_repealed']
                                           Act Title Act No Act Year  \
0     THE [***] WILLS AND INTESTACY REGULATION, 1799      V     1799   
1  THE  [***] CHARITABLE ENDOWMENTS PUBLIC BUILDI...    XIX     1810   
2         THE [***] STATE PRISONERS REGULATION, 1818    III     1818   

                                    Act Link  \
0  http://bdlaws.minlaw.gov.bd/act-1315.html   
1  http://bdlaws.minlaw.gov.bd/act-1316.html   
2  http://bdlaws.minlaw.gov.bd/act-1317.html   

                                         full_text  is_repealed  
0  http://bdlaws.minlaw.gov.bd/act-print-1315.html        False  
1  http://bdlaws.minlaw.gov.bd/act-print-1316.html        False  
2  http://bdlaws.minlaw.gov.bd/act-print-1317.html        False  


In [ ]:
EXCLUDE_REPEALED = True  # আপনার সিদ্ধান্ত অনুযায়ী True/False করুন

if EXCLUDE_REPEALED and "is_repealed" in act_index.columns:
    before = len(act_index)
    act_index = act_index[act_index["is_repealed"] == False]
    print(f"Excluded repealed acts: {before} -> {len(act_index)}")
else:
    print("Keeping all acts (repealed included).")


Excluded repealed acts: 1489 -> 1284


## ধাপ ৪: Act ID Resolve করুন

পরের সব ফিল্টারিং ধাপে (language filter, domain filter, section parsing) `act_id` লাগবে, তাই এখনই বের করে নিচ্ছি — `full_text` link থেকে।


In [ ]:
def extract_act_id_from_url(url):
    """e.g. http://bdlaws.minlaw.gov.bd/act-print-734.html -> 734"""
    m = re.search(r"act-print-(\d+)\.html", str(url))
    return m.group(1) if m else None

link_col = "full_text" if "full_text" in act_index.columns else None
if link_col is None:
    raise ValueError("Could not find the print-link column — check act_index.columns above")

act_index = act_index.copy()
act_index["act_id"] = act_index[link_col].apply(extract_act_id_from_url)
act_index = act_index.dropna(subset=["act_id"])

print(f"Acts with resolvable IDs: {len(act_index)}")


Acts with resolvable IDs: 1284


## ধাপ ৫: Language-Based Filtering — আগে ভাষা, তারপর Domain (গুরুত্বপূর্ণ ক্রম)

**টেস্ট রান করে যা ধরা পড়েছে:** যদি আগে ইংরেজি keyword দিয়ে domain filter করা হয়, তাহলে বেশিরভাগ পুরনো Colonial-era (ইংরেজি ভাষার) আইন বেছে নেওয়া হয়ে যায় — কারণ ঐ যুগের আইনগুলো মূলত ইংরেজিতেই প্রণীত এবং title-ও ইংরেজিতে। এরপর ভাষা-ফিল্টার লাগালে প্রায় সব বাদ পড়ে যায়।

যেহেতু আপনার focus **বাংলা** legal document-এ, তাই এখানে **আগে ভাষা দিয়ে filter করা হচ্ছে** (BLAD-এর act-level `language` field, পুরো corpus-এ verified ৬৯৮টা Act বাংলায় ট্যাগ করা), তারপরের ধাপে এই বাংলা Act গুলোর মধ্যেই domain-keyword খোঁজা হবে বাংলা শব্দ দিয়ে।


In [ ]:
FILTER_BY_LANGUAGE = True  # আপনার সিদ্ধান্ত অনুযায়ী বদলান
TARGET_LANGUAGE = "bengali"  # BLAD-এর act-level 'language' field-এ এই ভ্যালু ব্যবহার হয়েছে

if FILTER_BY_LANGUAGE:
    bangla_act_ids = []
    for _, row in act_index.iterrows():
        act_id = row["act_id"]
        fpath = os.path.join(ACTS_DIR, f"act-print-{act_id}.json")
        if not os.path.exists(fpath):
            continue
        with open(fpath, encoding="utf-8") as f:
            act = json.load(f)
        if act.get("language", "").lower() == TARGET_LANGUAGE:
            bangla_act_ids.append(act_id)

    before = len(act_index)
    act_index = act_index[act_index["act_id"].isin(bangla_act_ids)]
    print(f"Language filter applied ({TARGET_LANGUAGE}): {before} -> {len(act_index)} acts")
else:
    print("Language filter skipped — keeping all languages.")

working_index = act_index


Language filter applied (bengali): 1284 -> 698 acts


## ধাপ ৬: Domain-Based Filtering (Optional — বাংলা keyword দিয়ে, আপনার scope অনুযায়ী)

এখন উপরের বাংলা-ভাষার Act গুলোর মধ্যেই নির্দিষ্ট domain (criminal/family/labor/property) খুঁজব — **বাংলা keyword দিয়ে**, কারণ Act title বাংলায়।

সব বাংলা Act রাখতে চাইলে `USE_FOCUSED_SUBSET = False` করে দিন।


In [ ]:
USE_FOCUSED_SUBSET = True  # আপনার সিদ্ধান্ত অনুযায়ী বদলান

# উদাহরণ: অপরাধ/ফৌজদারি, নারী-শিশু, বিবাহ-তালাক-যৌতুক, শ্রম, সম্পত্তি-ভূমি
DOMAIN_KEYWORDS_BN = [
    "অপরাধ", "দণ্ডবিধি", "ফৌজদারি",
    "বিবাহ", "তালাক", "যৌতুক", "নারী", "শিশু", "পারিবারিক", "নির্যাতন",
    "শ্রম", "শ্রমিক",
    "সম্পত্তি", "ভূমি", "জমি", "উত্তরাধিকার",
]

if USE_FOCUSED_SUBSET:
    pattern = "|".join(DOMAIN_KEYWORDS_BN)
    focused_index = working_index[
        working_index["Act Title"].str.contains(pattern, case=False, na=False, regex=True)
    ]
    print(f"Focused subset: {len(focused_index)} acts (out of {len(working_index)} bangla acts)")
    print(focused_index["Act Title"].head(10).tolist())
    working_index = focused_index
else:
    print(f"Using all bangla acts: {len(working_index)}")


Focused subset: 49 acts (out of 698 bangla acts)
['ভূমি-খতিয়ান (পার্বত্য চট্টগ্রাম) অধ্যাদেশ, ১৯৮৪', 'অস্থাবর সম্পত্তি হুকুমদখল আইন, ১৯৮৮', 'ভূমি সংস্কার বোর্ড আইন, ১৯৮৯', 'ভূমি আপীল বোর্ড আইন, ১৯৮৯', 'আভ্যন্তরীণ নৌযান শ্রমিক (নিয়োগ-নিয়ন্ত্রণ) আইন, ১৯৯২', 'সন্ত্রাসমূলক অপরাধ দমন (বিশেষ বিধান) আইন, ১৯৯৪', 'যমুনা বহুমুখী সেতু প্রকল্প (ভূমি অধিগ্রহণ) আইন, ১৯৯৫', 'বিমান-নিরাপত্তা বিরোধী অপরাধ দমন আইন, ১৯৯৭', 'মহা হিসাব নিরীক্ষক ও নিয়ন্ত্রক (পারিশ্রমিক ও বিশেষ অধিকার) আইন, ১৯৯৮', 'নারী ও শিশু নির্যাতন দমন আইন, ২০০০']


## ধাপ ৭: Historical Range Decision (Optional)

Dataset-এ 1799 থেকে 2025 পর্যন্ত Act আছে। পুরনো Colonial-era আইনের ভাষা আধুনিক legal QA-র জন্য কম relevant হতে পারে।
নিচে স্বাধীনতা-পরবর্তী (1971+) Act-এ সীমাবদ্ধ রাখার option দেওয়া হলো — **এটা আপনার research design সিদ্ধান্ত, paper-এ justify করে লিখতে হবে।**


In [ ]:
RESTRICT_TO_POST_1971 = False  # True করলে শুধু 1971-এর পরের Act থাকবে

if RESTRICT_TO_POST_1971 and "Act Year" in working_index.columns:
    before = len(working_index)
    working_index = working_index[working_index["Act Year"] >= 1971]
    print(f"Restricted to post-1971: {before} -> {len(working_index)}")
else:
    print(f"Keeping full year range. Acts: {len(working_index)}")


Keeping full year range. Acts: 49


## ধাপ ৮: Selected Act গুলোর Full JSON Load করে Section Parse করুন

এখন `working_index`-এ যে Act গুলো বাছাই হয়েছে, সেগুলোর `act_id` মিলিয়ে আসল `acts/act-print-{id}.json` ফাইল থেকে full section text লোড করব, আর section number আলাদা করে বের করব (regex দিয়ে, কারণ raw text-এর শুরুতেই বাংলা সংখ্যায় embedded)।


In [ ]:
BN_DIGITS = "০১২৩৪৫৬৭৮৯"
EN_DIGITS = "0123456789"
BN_TO_EN = str.maketrans(BN_DIGITS, EN_DIGITS)

import re

def parse_section(section_content):
    text = section_content.strip()
    # দুই ধরনের দাঁড়িই মেলান: । (U+0964) এবং ৷ (U+09F7)
    m = re.match(r"^([০-৯]+)[।৷]", text)
    if m:
        bn_num = m.group(1)
        en_num = bn_num.translate(BN_TO_EN)
        return en_num, text
    return None, text


def clean_text(text):
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def detect_language(text):
    bangla_chars = len(re.findall(r"[\u0980-\u09FF]", text))
    latin_chars = len(re.findall(r"[A-Za-z]", text))
    if bangla_chars > latin_chars * 2:
        return "bn"
    elif latin_chars > bangla_chars * 2:
        return "en"
    else:
        return "mixed"

print("Parsing helper functions ready.")


Parsing helper functions ready.


In [ ]:
processed_acts = []
skipped = []

for _, row in working_index.iterrows():
    act_id = row["act_id"]
    fname = f"act-print-{act_id}.json"
    fpath = os.path.join(ACTS_DIR, fname)

    if not os.path.exists(fpath):
        skipped.append(act_id)
        continue

    with open(fpath, encoding="utf-8") as f:
        raw_act = json.load(f)

    sections_out = []
    for sec in raw_act.get("sections", []):
        raw_text = sec.get("section_content", "")
        if not raw_text:
            continue
        sec_num, sec_text = parse_section(raw_text)
        sec_text = clean_text(sec_text)
        sections_out.append({
            "section_number": sec_num,
            "section_text": sec_text,
            "language": detect_language(sec_text),
        })

    processed_acts.append({
        "act_id": act_id,
        "act_title": raw_act.get("act_title"),
        "act_no": raw_act.get("act_no"),
        "act_year": raw_act.get("act_year"),
        "source_url": raw_act.get("source_url"),
        "token_count": raw_act.get("token_count"),
        "sections": sections_out,
    })

print(f"Processed acts: {len(processed_acts)}")
print(f"Skipped (file not found): {len(skipped)}")


Processed acts: 49
Skipped (file not found): 0


## ধাপ ৯: Corpus Statistics (এটাই আপনার Paper-এর Table 1 হবে)


In [ ]:
total_acts = len(processed_acts)
total_sections = sum(len(a["sections"]) for a in processed_acts)
total_tokens = sum(a.get("token_count") or 0 for a in processed_acts)

lang_counter = Counter()
for a in processed_acts:
    for s in a["sections"]:
        lang_counter[s["language"]] += 1

year_values = [int(a["act_year"]) for a in processed_acts if str(a.get("act_year")).isdigit()]
year_min = min(year_values) if year_values else None
year_max = max(year_values) if year_values else None

print("=" * 50)
print("CORPUS STATISTICS (BDLegalRAG — corpus layer)")
print("=" * 50)
print(f"Total Acts:            {total_acts}")
print(f"Total Sections:        {total_sections}")
print(f"Total Tokens (approx): {total_tokens}")
print(f"Year Range:            {year_min} - {year_max}")
print(f"Language Distribution: {dict(lang_counter)}")
print("=" * 50)


CORPUS STATISTICS (BDLegalRAG — corpus layer)
Total Acts:            49
Total Sections:        1575
Total Tokens (approx): 183635
Year Range:            1985 - 2025
Language Distribution: {'bn': 1561, 'mixed': 13, 'en': 1}


## ধাপ ১০: Final Cleaned Corpus Save করুন (Checkpoint)

এই ফাইলটাই পরের ধাপ — **Stage 3: Section-Aware Chunking Experiment**-এ ব্যবহার হবে।


In [ ]:
import os # Ensure os is imported, though it likely is from earlier cells.
output_path = os.path.join(PROCESSED_DIR, "bdlegalrag_corpus_v1.json")

# Ensure the directory exists before attempting to write the file
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump({
        "meta": {
            "source": "BLAD (Bangladesh Legal Acts Dataset)",
            "source_citation": "Sakhawat, A. (2025). BLAD: A Historically Contextualized, "
                                "Multilingual Dataset of Bangladeshi Legal Acts (1799 to 2025). "
                                "arXiv:2607.17111.",
            "excluded_repealed": EXCLUDE_REPEALED,
            "focused_subset": USE_FOCUSED_SUBSET,
            "domain_keywords_bn": DOMAIN_KEYWORDS_BN if USE_FOCUSED_SUBSET else None,
            "language_filtered": FILTER_BY_LANGUAGE,
            "target_language": TARGET_LANGUAGE if FILTER_BY_LANGUAGE else None,
            "post_1971_only": RESTRICT_TO_POST_1971,
            "total_acts": total_acts,
            "total_sections": total_sections,
            "total_tokens": total_tokens,
            "year_range": [year_min, year_max],
            "language_distribution": dict(lang_counter),
        },
        "acts": processed_acts,
    }, f, ensure_ascii=False, indent=2)

print("Saved cleaned corpus to:", output_path)
print("File size (MB):", round(os.path.getsize(output_path) / 1e6, 2))


Saved cleaned corpus to: /content/drive/MyDrive/BDLegalRAG/processed_data/bdlegalrag_corpus_v1.json
File size (MB): 3.13


## পরের ধাপ

এই notebook-এর `processed_data/bdlegalrag_corpus_v1.json` ফাইলটাই এখন আপনার corpus layer-এর deliverable।

**পরবর্তী notebook-এ যা করবেন (Stage 3 — Chunking Experiment):**
1. এই JSON load করুন
2. তিনটা chunking strategy try করুন: Fixed-size (256/512 token), Recursive splitter, Section-aware (already section-level এখানে, তাই এখান থেকেই শুরু)
3. প্রতিটার জন্য retrieval evaluation (Precision@k, Recall@k) চালান

চাইলে আমি এখনই Stage 3-এর notebook-ও বানিয়ে দিতে পারি — বলুন করে দিই কিনা।
